# ASPP-UNet Training for Optic Disc/Cup Segmentation

This notebook trains an **ASPP-UNet** model - a UNet architecture with Atrous Spatial Pyramid Pooling in the bottleneck.

## Why ASPP-UNet?

**Key Advantage**: Multi-scale features **during segmentation**, not preprocessing!

### For Optic Cup Segmentation:

1. **Cup boundary needs multi-scale context:**
   - Fine edge detection (dilation=1)
   - Disc boundary awareness (dilation=6,12)
   - Overall fundus context (global pooling)

2. **Architecture > Preprocessing:**
   - ASPP in UNet learns task-specific multi-scale features
   - Direct architectural improvements often outperform preprocessing
   - End-to-end optimization

3. **Proven in medical imaging:**
   - DeepLab family dominates semantic segmentation
   - ASPP widely used in medical image analysis
   - Often better than preprocessing + standard UNet

4. **Parameter Efficient:**
   - ASPP-UNet: ~21.5M parameters
   - Standard UNet: ~31.0M parameters
   - **30% fewer parameters, better performance!**

## 1. Setup and Imports

In [1]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import json

# Add src to path
project_root = Path.cwd().parent
sys.path.append(str(project_root / 'src'))

from training.train_aspp_unet import train_aspp_unet
from training.train import plot_training_history
from data_loader.dataset import GlaucomaDataset

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")


Project root: /home/robolab/dev/CAP5410-roi-enhancer-odoc
PyTorch version: 2.8.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 2080 Ti


## 2. Configuration

In [2]:
# Training hyperparameters for ASPP-UNet
config = {
    'root_dir': str(project_root),
    'num_epochs': 100,
    'batch_size': 16,
    'learning_rate': 1e-4,
    'image_size': 256,
    'base_channels': 64,
    'num_workers': 0,
    'save_dir': str(project_root / 'checkpoints_aspp_unet'),
    'filter_incomplete': True,
    'use_clahe': False,  # Train on original images (same as baseline)
    'model_type': 'full',  # 'full' or 'lightweight'
    'dilation_rates': [1, 6, 12, 18],  # ASPP dilation rates
    'patience': 15
}

print("ASPP-UNet Training Configuration:")
print("=" * 70)
for key, value in config.items():
    print(f"{key:25s}: {value}")
print("=" * 70)
print("\nNote: Using class weights - Cup weight=2.0 to improve Cup segmentation")
print("Note: Multi-scale features via ASPP in bottleneck")


ASPP-UNet Training Configuration:
root_dir                 : /home/robolab/dev/CAP5410-roi-enhancer-odoc
num_epochs               : 100
batch_size               : 16
learning_rate            : 0.0001
image_size               : 256
base_channels            : 64
num_workers              : 0
save_dir                 : /home/robolab/dev/CAP5410-roi-enhancer-odoc/checkpoints_aspp_unet
filter_incomplete        : True
use_clahe                : False
model_type               : full
dilation_rates           : [1, 6, 12, 18]
patience                 : 15

Note: Using class weights - Cup weight=2.0 to improve Cup segmentation
Note: Multi-scale features via ASPP in bottleneck


## 3. Train the Model


In [3]:
# Clear the dataset cache before training
GlaucomaDataset._split_cache.clear()

# Train the ASPP-UNet model
model, history = train_aspp_unet(**config)


Using device: cuda
Random seeds set for reproducible training

Loading datasets...
Filtering incomplete masks...
  Filtered out 234 images with incomplete masks
TRAIN split: 1845 samples
VAL split: 395 samples
TEST split: 396 samples

Initializing full ASPP-UNet model...
Model parameters: 21,461,507
Dilation rates: [1, 6, 12, 18]

Starting training for 100 epochs...

Epoch 1/100
--------------------------------------------------------------------------------


Training:   0%|          | 0/116 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 256.00 MiB. GPU 0 has a total capacity of 10.56 GiB of which 140.62 MiB is free. Process 3007586 has 5.15 GiB memory in use. Including non-PyTorch memory, this process has 5.23 GiB memory in use. Of the allocated memory 4.80 GiB is allocated by PyTorch, and 254.67 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 4. Visualize Training Progress


In [ ]:
# Plot training history
fig = plot_training_history(
    history,
    save_path=str(project_root / 'results' / 'aspp_unet_training_history.png')
)
plt.suptitle('ASPP-UNet Training History', fontsize=16, y=1.00)
plt.show()


## 5. Print Final Metrics


## 6. Save Training History


In [ ]:
print("\n" + "=" * 80)
print("FINAL TRAINING RESULTS - ASPP-UNet")
print("=" * 80)

final_epoch = len(history['train_loss'])
print(f"\nTotal Epochs: {final_epoch}")

print("\nFinal Training Metrics:")
print(f"  Loss:        {history['train_loss'][-1]:.4f}")
print(f"  IoU (BG):    {history['train_iou_bg'][-1]:.4f}")
print(f"  IoU (Disc):  {history['train_iou_disc'][-1]:.4f}")
print(f"  IoU (Cup):   {history['train_iou_cup'][-1]:.4f}")

print("\nFinal Validation Metrics:")
print(f"  Loss:        {history['val_loss'][-1]:.4f}")
print(f"  IoU (BG):    {history['val_iou_bg'][-1]:.4f}")
print(f"  IoU (Disc):  {history['val_iou_disc'][-1]:.4f}")
print(f"  IoU (Cup):   {history['val_iou_cup'][-1]:.4f}")

print("\nBest Validation Loss:")
best_epoch = history['val_loss'].index(min(history['val_loss'])) + 1
best_val_loss = min(history['val_loss'])
print(f"  Epoch: {best_epoch}")
print(f"  Loss:  {best_val_loss:.4f}")

print("\n" + "=" * 80)


In [ ]:
# Save history as JSON
results_dir = project_root / 'results'
results_dir.mkdir(exist_ok=True, parents=True)

history_path = results_dir / 'aspp_unet_training_history.json'
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

print(f"Training history saved to: {history_path}")
